# Wasserstein Ascend Descend
### (C. and N.) Garcia Trillos

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import sys
import numpy as np
from Robust_nn.WAD import WAD2scale
import matplotlib.pyplot as plt
from utils.utils import read_vision_dataset
from utils.convnet import ConvNet
from utils.convnet_silu import ConvNetSiLU
import os
import torch
import gc
import torch.nn as nn
from torch.optim.swa_utils import AveragedModel, SWALR


**Read the DataLoader**

In [41]:
# Create some networks

# n_nets = 5
n_nets = 2
net_lst = [ConvNet() for i in range(n_nets)]
avg_nets =[ConvNet() for i in range(n_nets*4)]

# adv_net = WAD2scale(net_list = net_lst, avg_nets = avg_nets, 
#                     dataset_name='MNIST',batch_size = 1024, 
#                     device = None , criterion= nn.CrossEntropyLoss(), 
#                     scale_factor=5, num_adverse=2,
#                     kappa = { 'param': 0.2, 'adv': 0.2   },
#                     max_batches= 5)


adv_net = WAD2scale(net_list = net_lst, avg_nets = avg_nets, 
                    dataset_name='MNIST',batch_size = 128, 
                    device = None , criterion= nn.CrossEntropyLoss(), 
                    scale_factor=2, num_adverse=2,
                    kappa = { 'param': 0.2, 'adv': 0.2   },
                    max_batches= 2)

In [42]:
import os
os.listdir('.')

['.git',
 '.vscode',
 'checkpoint',
 'Robust_nn',
 'Robust_reg_code.zip',
 'Test1_kappa0p2-3',
 'Test1_kappa0p2-4',
 'Tests_o1o2.ipynb',
 'Tests_WAD.ipynb',
 'Tests_WAD_script.py',
 'todo.md',
 'utils',
 '_avg_adv_folder']

In [43]:
adv_net.set_optimizer()
adv_net.train(epochs = 10)


Epoch: 0
tensor([0.5000, 0.5000])
tensor([0.5000, 0.5000])
0|1|2|
Epoch: 1
tensor([0.5006, 0.4994])
tensor([0.5014, 0.4986], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 2
tensor([0.5000, 0.5000])
tensor([0.5018, 0.4982], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 3
tensor([0.4989, 0.5011])
tensor([0.5033, 0.4967], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 4
tensor([0.4986, 0.5014])
tensor([0.5056, 0.4944], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 5
tensor([0.4979, 0.5021])
tensor([0.5082, 0.4918], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 6
tensor([0.4968, 0.5032])
tensor([0.5121, 0.4879], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 7
tensor([0.4961, 0.5039])
tensor([0.5169, 0.4831], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 8
tensor([0.4956, 0.5044])
tensor([0.5228, 0.4772], grad_fn=<DivBackward0>)
0|1|2|
Epoch: 9
tensor([0.4940, 0.5060])
tensor([0.5300, 0.4700], grad_fn=<DivBackward0>)
0|1|2|

In [30]:
adv_net.save_model('Test1_kappa0p2-4')

Saving...
done.


In [31]:
adv_net.set_optimizer()
adv_net.import_model('Test1_kappa0p2-4')

Loading...
done.


In [32]:
'''
Testing the model 

Peform the following tests:
1. Test for accuracy of the average  model
2. Train directly the model with the whole set of adversaries. Calculate...
3. Create directly the adversarial. 
'''

'\nTesting the model \n\nPeform the following tests:\n1. Test for accuracy of the average  model\n2. Train directly the model with the whole set of adversaries. Calculate...\n3. Create directly the adversarial. \n'

In [33]:
adv_net.test_pgd(5)

epoch = 5, adv_acc = 24.41, clean_acc = 61.18
Saving the best model to ./checkpoint
Saving...
done.


(1.945834626125384, 24.41, 61.18)

In [44]:
adv_net.test_improve_model(5)

0|1|2|

AttributeError: 'list' object has no attribute 'shape'

In [35]:
adv_net.test_improve_adversaries(5)

RuntimeError: One of the differentiated Tensors does not require grad

**Creating and comparing results**

[tensor([0.1745, 0.3892, 0.6346, 0.0489, 0.0594]), tensor([0.0834, 0.4240, 0.5160, 0.3686, 0.5654]), tensor([0.6288, 0.8525, 0.3783, 0.9661, 0.8952])]
tensor([0.8867, 1.6657, 1.5288, 1.3836, 1.5201])


In [ ]:
# options = {'only o1':(True,False, False), 'only o2':(False, True, False), 'both': (True,True, False), 'none':(False,False,False), 'modO2':(False,True,True) } 
options = {'only o1':(True,False, False), 'both': (True,True, True), 'none':(False,False,False) } 
rvec = [2,4,np.inf]
deltav = [0.2]
mdict = {}
basepath = os.curdir
mpath = os.path.join(basepath,'models')

for i,(k,(o1,o2, mod_o2)) in enumerate(options.items()):
  auxd = {}
  for r in rvec:
    rstr = str(r)
    print('\n*******\n ** Case '+k+', r=',r,'\n ******')
    torch.manual_seed(0)
    np.random.seed(0)
    auxd[rstr] = {}
    network = ConvNet()
    net_RegTrin = OrdTwoL(network, trainloader, testloader,  device='cuda', delta =0.2, r= r, o2=o2, o1=o1, mod_o2=mod_o2) #'cuda'
    net_RegTrin.set_optimizer(optim_alg='Adam', args={'lr':1e-4})
    net_RegTrin.train(epochs=5, delta=deltav)
    auxd[rstr]['train_loss'] = net_RegTrin.train_loss.copy()
    auxd[rstr]['train_acc'] = net_RegTrin.train_acc.copy()
    auxd[rstr]['train_reg'] = net_RegTrin.train_reg.copy()
    auxd[rstr]['test_acc_adv'] = net_RegTrin.test_acc_adv.copy()
    auxd[rstr]['test_acc_clean'] = net_RegTrin.test_acc_clean.copy()
    auxd[rstr]['train_times'] = net_RegTrin.train_times.copy()
    torch.save(network.state_dict(), os.path.join(mpath, k+'_r_'+str(r)+'.pth' ))
    del(network)
    gc.collect()
    torch.cuda.empty_cache()
  mdict[k] = auxd
  with open('tests_all.txt','w') as data: 
      data.write(str(mdict))